# 02 · State, Nodes and Edges: How the Runtime Really Works

> **Kernel:** `Phase 5 · LangGraph` · **Runs offline**: no model needed. The
> nodes are plain functions, so you can watch the machinery itself.

Notebook 01 ran simple graphs. This one opens the hood. Designing the **state**
well, knowing how **updates are merged**, and understanding **when nodes run**
decide whether a LangGraph program is correct. Most LangGraph bugs are really
misunderstandings of this notebook's material.

**You will learn**
- How the runtime executes a graph in **supersteps**, and why that makes branches parallel
- **Reducers**: how updates are merged, and what LangGraph does when two branches write the same key
- **Conditional edges** (routers), including routing to several nodes at once
- The **fan-in trap** with uneven branches, and the two ways to fix it
- State schemas: `TypedDict` vs Pydantic, private keys, input/output schemas
- Two silent gotchas: unknown keys are dropped, and Pydantic doesn't validate node outputs

**Prerequisites:** notebook 01. *Optional deeper background:* `00_CONCEPTS/02`
builds this same runtime by hand in about 120 lines.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import operator, time
from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, START, END

## 1 · Why this matters

The state is the **only** way nodes communicate, and the only thing a
checkpoint saves. So the state schema is the contract of your whole program:

- If two parallel branches write the same key, *something* must decide how the
  values combine. Otherwise data is lost or the run fails.
- If a node's work isn't in the state, it can't be checkpointed, resumed,
  inspected, or shown to a human.
- If the state holds things that can't be serialised (clients, open files),
  checkpointing breaks.

## 2 · How the runtime runs a graph: supersteps

LangGraph's runtime (named *Pregel*, after the Google system that inspired it)
executes in rounds called **supersteps**:

```
 ┌──────────────────────────── one superstep ────────────────────────────┐
 │ 1. find every node that is due  (its incoming edge fired last round)  │
 │ 2. run them ALL, in parallel, each on a snapshot of the state         │
 │ 3. collect their updates and merge them into the state (via reducers) │
 │ 4. follow their outgoing edges → the nodes due in the next superstep  │
 └────────────────────────────────────────────────────────────────────────┘
      repeat until nothing is due  →  return the final state
      (with a checkpointer: save the state after every superstep)
```

Two consequences to remember:
- Nodes in the **same** superstep run **concurrently** and can't see each
  other's updates. They all read the state as it was at the start of the round.
- A node that several branches point to runs in the **next** round, *once*,
  if all those branches finished in the same round.

Let's see it. Three nodes each sleep 1 second and fan out from `START`:

In [ ]:
class Research(TypedDict):
    notes: Annotated[list[str], operator.add]     # ← a reducer (explained in §3)
    report: str


def fetcher(topic):
    def fetch(state: Research) -> dict:
        time.sleep(1.0)                            # pretend: a slow API
        return {'notes': [f'notes on {topic}']}
    fetch.__name__ = f'fetch_{topic}'
    return fetch

def write(state: Research) -> dict:
    return {'report': ' | '.join(sorted(state['notes']))}


builder = StateGraph(Research)
for topic in ('batteries', 'permits', 'software'):
    builder.add_node(f'fetch_{topic}', fetcher(topic))
    builder.add_edge(START, f'fetch_{topic}')       # fan-out: three edges from START
    builder.add_edge(f'fetch_{topic}', 'write')     # fan-in:  three edges into write
builder.add_node('write', write)
builder.add_edge('write', END)
research = builder.compile()

print(research.get_graph().draw_ascii())
t0 = time.perf_counter()
for step, update in enumerate(research.stream({'notes': []}, stream_mode='updates'), 1):
    print(f'{time.perf_counter() - t0:4.1f}s  {update}')

The three fetches finished together at ~1.0s: one superstep, run in parallel,
even though they're ordinary synchronous functions (LangGraph runs sync nodes in
a thread pool). `write` then ran **once**, in the next superstep, and saw all
three notes.

## 3 · Reducers: how updates are merged

When a node returns `{'key': value}`, the runtime has to decide how `value`
combines with what's already in the state. That decision is the key's
**reducer**:

| Declared as | Reducer | Result of an update |
|---|---|---|
| `report: str` | *(none)* → **overwrite** | the new value replaces the old one |
| `notes: Annotated[list[str], operator.add]` | `operator.add` | `old + new`: lists are concatenated |
| `scores: Annotated[dict, merge_dicts]` | your own function `(old, new) -> merged` | whatever you define |

`Annotated[type, reducer]` is standard Python typing: it attaches extra
information (here, the reducer) to a type. LangGraph reads it from your schema.

**What if two parallel branches write a key that has *no* reducer?** Many
engines just let the last write win and silently lose data. LangGraph refuses:

In [ ]:
class NoReducer(TypedDict):
    notes: list[str]                   # ← no reducer: overwrite semantics


b = StateGraph(NoReducer)
for topic in ('batteries', 'permits'):
    b.add_node(f'fetch_{topic}', lambda s, t=topic: {'notes': [t]})
    b.add_edge(START, f'fetch_{topic}')
try:
    b.compile().invoke({'notes': []})
except Exception as err:
    print(type(err).__name__, '→', str(err).splitlines()[0])

`InvalidUpdateError`: *"Can receive only one value per step"*. LangGraph
detects that two updates to an overwrite-only key arrived in the same superstep
and stops, instead of silently picking one. **Rule: every key that parallel
branches write needs a reducer.**

A **custom reducer** is any function `(current, update) -> new`. A common one
merges dictionaries, so parallel reviewers can each add their own score:

In [ ]:
def merge_dicts(current: dict | None, update: dict) -> dict:
    return {**(current or {}), **update}


class Review(TypedDict):
    draft: str
    scores: Annotated[dict[str, int], merge_dicts]


def reviewer(name, score):
    def review(state: Review) -> dict:
        return {'scores': {name: score}}         # each branch adds its own key
    review.__name__ = name
    return review


b = StateGraph(Review)
for name, score in [('clarity', 8), ('accuracy', 6), ('tone', 9)]:
    b.add_node(name, reviewer(name, score))
    b.add_edge(START, name)
print(b.compile().invoke({'draft': 'v1', 'scores': {}}))

**Bypassing a reducer.** Sometimes a node needs to *replace* a reduced key, for
example to reset an accumulated list. Wrap the value in `Overwrite(...)` and the
reducer is skipped for that one update:

In [ ]:
from langgraph.types import Overwrite

b = StateGraph(Research)
b.add_node('reset', lambda s: {'notes': Overwrite(['fresh start'])})
b.add_edge(START, 'reset')
print(b.compile().invoke({'notes': ['old 1', 'old 2']}))

## 4 · Conditional edges (routers)

A **conditional edge** calls a function after a node finishes. The function
reads the state and returns *where to go next*:

```python
builder.add_conditional_edges('source', router_fn, path_map)
```

- `router_fn(state)` returns a node name, `END`, or a **list** of names (all run
  in parallel next superstep).
- `path_map` lists the possible destinations. It's optional, but include it:
  it documents the graph, lets `draw_ascii()` show the branches, and catches
  typos at compile time.

A support-ticket triage: urgent billing tickets go to **both** the refund desk
and a human supervisor, in parallel.

In [ ]:
class Ticket(TypedDict):
    text: str
    category: str
    urgent: bool
    actions: Annotated[list[str], operator.add]


def classify(state: Ticket) -> dict:
    t = state['text'].lower()
    return {'category': 'billing' if 'charge' in t or 'refund' in t else 'tech',
            'urgent': 'urgent' in t or 'twice' in t}

def route(state: Ticket) -> list[str] | str:
    if state['category'] == 'billing':
        return ['refund_desk', 'supervisor'] if state['urgent'] else 'refund_desk'
    return 'tech_support'

def refund_desk(state):  return {'actions': ['refund opened']}
def supervisor(state):   return {'actions': ['supervisor paged']}
def tech_support(state): return {'actions': ['troubleshooting sent']}


b = StateGraph(Ticket)
for fn in (classify, refund_desk, supervisor, tech_support):
    b.add_node(fn.__name__, fn)
b.add_edge(START, 'classify')
b.add_conditional_edges('classify', route, ['refund_desk', 'supervisor', 'tech_support'])
for n in ('refund_desk', 'supervisor', 'tech_support'):
    b.add_edge(n, END)
triage = b.compile()

print(triage.get_graph().draw_ascii())
for text in ['I was charged twice, urgent!', 'Please refund my March invoice', 'App crashes on login']:
    out = triage.invoke({'text': text, 'actions': []})
    print(f'{text:<32} → {out["actions"]}')

The router is plain code here. Replace its body with an LLM call returning
structured output and you have "let the model decide *here*", with the model
limited to the destinations you listed. Notebook 05 does exactly that.

## 5 · The fan-in trap: branches of different length

In §2 the three branches were the same length, so `write` ran once. Now make one
branch longer: `a → a2 → join` and `b → join`.

In [ ]:
runs = []

def node(name):
    def fn(state: Research) -> dict:
        runs.append(name)
        return {'notes': [name]}
    fn.__name__ = name
    return fn


def build(join_style):
    b = StateGraph(Research)
    for n in ('a', 'a2', 'b'):
        b.add_node(n, node(n))
    b.add_node('join', node('join'), defer=(join_style == 'defer'))
    b.add_edge(START, 'a')
    b.add_edge(START, 'b')
    b.add_edge('a', 'a2')
    if join_style == 'wait_for_all':
        b.add_edge(['a2', 'b'], 'join')          # ONE edge that waits for both
    else:
        b.add_edge('a2', 'join')
        b.add_edge('b', 'join')
    b.add_edge('join', END)
    return b.compile()


for style in ('naive', 'wait_for_all', 'defer'):
    runs.clear()
    build(style).invoke({'notes': []})
    print(f'{style:<13} join ran {runs.count("join")}×   order: {runs}')

What happened in the **naive** version, superstep by superstep:

```
 step 1:  a, b            (both from START)
 step 2:  a2, join  ← b finished in step 1, so its edge fired: join runs EARLY, missing a2's work
 step 3:  join      ← a2 finished in step 2: join runs AGAIN
```

Two fixes:
- **`add_edge(['a2', 'b'], 'join')`**: a single edge from a *list* of nodes
  means "run `join` when **all** of these have finished". Use it when you know
  exactly which branches must complete.
- **`add_node('join', ..., defer=True)`**: a deferred node waits until every
  other pending branch of the run has finished. Use it when the set of branches
  varies (e.g. conditional or dynamic fan-out).

## 6 · Choosing a state schema

LangGraph accepts a `TypedDict`, a `dataclass`, or a Pydantic `BaseModel`.

| | `TypedDict` | Pydantic `BaseModel` |
|---|---|---|
| Access in nodes | `state['key']` | `state.key` |
| Defaults | no | yes |
| Validation | none | on the graph **input** only (see below) |
| Speed | fastest | slower (validation on every node input) |
| Typical use | most graphs | graphs taking untrusted input |

Pydantic validates what you pass to `invoke`:

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Order(BaseModel):
    quantity: int = Field(ge=1)
    status: str = 'new'


b = StateGraph(Order)
b.add_node('approve', lambda s: {'status': f'approved {s.quantity} units'})
b.add_edge(START, 'approve')
orders = b.compile()

print(orders.invoke({'quantity': 3}))
try:
    orders.invoke({'quantity': 0})
except ValidationError as err:
    print('rejected input →', err.errors()[0]['msg'])

**Gotcha 1: node outputs are *not* validated.** A node can write a value that
breaks your own schema, and nothing complains:

In [ ]:
b = StateGraph(Order)
b.add_node('bug', lambda s: {'quantity': -5})          # violates ge=1
b.add_edge(START, 'bug')
print('accepted without error →', b.compile().invoke({'quantity': 3}))

**Gotcha 2: unknown keys are dropped silently.** If a node returns a key that
isn't in the schema, for example after a typo, the update vanishes:

In [ ]:
b = StateGraph(Research)
b.add_node('typo', lambda s: {'reprot': 'the final report'})   # 'reprot' ≠ 'report'
b.add_edge(START, 'typo')
print(b.compile().invoke({'notes': []}), ' ← where did the report go?')

Both gotchas have the same defence: **unit-test your nodes** (they're plain
functions) and assert on the keys and values they return. Notebook 12 shows a
testing pattern.

## 7 · Private state: input and output schemas

Often a graph needs internal working keys that callers shouldn't pass in or see
come back: a scratchpad, intermediate drafts, retry counters. Give the graph
an **input schema** and an **output schema**, while the full state stays internal:

In [ ]:
class Question(TypedDict):          # what callers send
    question: str

class Answer(TypedDict):            # what callers get back
    answer: str

class Internal(TypedDict):          # everything the nodes work with
    question: str
    scratchpad: list[str]
    answer: str


def think(state: Internal) -> dict:
    return {'scratchpad': ['considered option A', 'rejected option B']}

def respond(state: Internal) -> dict:
    return {'answer': f"{state['question']} → chose A after {len(state['scratchpad'])} thoughts"}


b = StateGraph(Internal, input_schema=Question, output_schema=Answer)
b.add_node('think', think)
b.add_node('respond', respond)
b.add_edge(START, 'think')
b.add_edge('think', 'respond')
b.add_edge('respond', END)
print(b.compile().invoke({'question': 'Which database?'}))

The caller sent only `question` and got back only `answer`. `scratchpad`
existed during the run (and would be in checkpoints) but isn't part of the
graph's public interface. That separation makes graphs much easier to compose
(notebook 10: subgraphs).

## 8 · Designing good state: guidelines

- **Store data, not resources.** Keep strings, numbers, lists and dicts in
  state, and keep clients, connections and file handles outside it (module
  level, or passed in through runtime context). Everything in state gets
  checkpointed.
- **Store facts, not formatted text.** Keep `{'score': 7, 'issues': [...]}`
  rather than a paragraph. Downstream nodes and routers can act on structure.
- **One writer per key, or a reducer.** If you can't name the single node that
  owns a key, it needs a reducer.
- **Keep it lean.** Large documents belong in files or a database, with the path
  or id in state. Every checkpoint copies the state.
- **Name keys for what they mean** (`approved_by`, `revision_count`), not for
  the node that writes them.

## 9 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Parallel writes to a key without a reducer | `InvalidUpdateError` | `Annotated[..., reducer]` |
| `operator.add` on a key you meant to replace | lists grow every loop iteration | no reducer, or `Overwrite(...)` |
| Uneven branches into one join | join runs twice, first time with partial data | `add_edge([...], join)` or `defer=True` |
| Typo in a returned key | the update silently disappears | unit-test node outputs |
| Relying on Pydantic to validate node outputs | invalid values flow on | validate inside the node, or test |
| Router returns a name not in `path_map` | error at runtime | keep `path_map` in sync; use `Literal` return types |
| Mutating `state` in place | changes lost or not checkpointed | always *return* updates |

## 10 · Check yourself

<details><summary><b>1.</b> Nodes <code>x</code> and <code>y</code> run in the same superstep. <code>x</code> writes <code>count=1</code>. Does <code>y</code> see it?</summary>

No. Nodes in one superstep all read the state as it was at the start of that step, and their updates are merged only after all of them finish. `y` sees the new `count` only if it runs in a later superstep.
</details>

<details><summary><b>2.</b> Why does LangGraph raise <code>InvalidUpdateError</code> instead of just keeping the last value?</summary>

Because with parallel branches "last" is arbitrary, and keeping one value silently discards the others. Raising forces you to declare the intended merge (a reducer), which turns a silent data-loss bug into a loud, fixable one.
</details>

<details><summary><b>3.</b> A loop node appends to <code>notes</code> (<code>operator.add</code>) on every iteration, but you want only the latest iteration's notes. What are your options?</summary>

Remove the reducer (plain overwrite) if only one node writes the key; or return `Overwrite([...])` from the loop node to replace the list; or store per-iteration notes under a separate key.
</details>

<details><summary><b>4.</b> Explain why <code>join</code> ran twice in §5, and when you'd choose <code>defer=True</code> over <code>add_edge([...], 'join')</code>.</summary>

Each incoming edge fires when its source finishes. `b` finished in step 1 and `a2` in step 2, so `join` was triggered twice. The list edge waits for a *known* set of sources; `defer=True` waits for all pending work, which is better when the branches are conditional or created dynamically.
</details>

<details><summary><b>5.</b> Name two things Pydantic state does and doesn't protect you from.</summary>

It validates and coerces the graph's input (types, constraints, defaults). It does **not** validate what nodes return, and it doesn't stop updates to unknown keys from being dropped.
</details>

<details><summary><b>6.</b> Why use separate input/output schemas?</summary>

To keep internal working keys (scratchpads, counters, drafts) out of the graph's public interface. Callers send only what's needed and get back only the result, which makes graphs easier to reuse and nest as subgraphs.
</details>

## Takeaway

The runtime works in **supersteps**: run every due node in parallel, merge their
updates through **reducers**, follow **edges** to the next set. Declare a reducer
for any key that parallel branches share, route with conditional edges, use
list edges or `defer=True` to join uneven branches, and test your nodes, because
typos and invalid values in updates pass silently.

Next → `03_chat_models_and_messages.ipynb`: talking to Ollama, OpenAI and
Anthropic through one interface, message types, and structured output.